# Notebook 7: OOD/OE Quality Human Review

This notebook audits prepared runtime dataset `ood/` and `oe/` pools, then leaves cleanup decisions to a human reviewer.

It checks:
- exact SHA-256 overlap across `continual`, `val`, `test`, `ood`, and `oe`
- perceptual near-duplicate candidates involving `ood/` or `oe/`
- path/slice naming suspicions for OOD/OE folders

The notebook does not choose quarantine rows for you. Edit `review_decisions.csv` manually and set only confirmed rows to `decision=quarantine`; the apply cell moves only those targets into `.runtime_tmp/ood_oe_quality_quarantine/`.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
CLONE_TARGET = Path(os.environ.get('AADS_REPO_CLONE_TARGET', '/content/bitirmeprojesi'))


def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'scripts').is_dir() and (path / 'config').is_dir()


def _resolve_repo_root() -> Path:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if raw:
            candidate = Path(raw).expanduser().resolve()
            if _is_repo_root(candidate):
                return candidate
    for candidate in [Path.cwd(), *Path.cwd().parents, CLONE_TARGET, Path('/content/bitirmeprojesi')]:
        candidate = candidate.expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if _is_repo_root(CLONE_TARGET):
        return CLONE_TARGET.resolve()
    raise FileNotFoundError('Repo root not found. Set AADS_REPO_ROOT or check the clone target.')


REPO_ROOT = _resolve_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f'[SETUP] repo={REPO_ROOT}')


In [ ]:
# Parameters
RUN_ALL_DATASETS = True
ADAPTER_KEY = 'tomato__leaf'
REVIEW_ADAPTER_KEY = ADAPTER_KEY

# Batch mode audits every prepared runtime dataset that has ood/ or oe/.
PREPARED_ROOT = REPO_ROOT / 'data' / 'prepared_runtime_datasets'

# Single-dataset mode uses this path when set; otherwise it uses PREPARED_ROOT / ADAPTER_KEY.
DATASET_ROOT = ''

# Audit artifacts are written here. Single-dataset mode writes under <OUTPUT_DIR>/<ADAPTER_KEY>.
OUTPUT_DIR = ''

# Conservative near-duplicate candidate threshold. Higher values create more review rows.
NEAR_DUPLICATE_HAMMING = 6

dataset_root = Path(DATASET_ROOT or (PREPARED_ROOT / ADAPTER_KEY)).expanduser()
output_dir = Path(OUTPUT_DIR or (REPO_ROOT / '.runtime_tmp' / 'ood_oe_quality_audit')).expanduser()
if not RUN_ALL_DATASETS:
    output_dir = output_dir / ADAPTER_KEY

print(f'[PARAM] run_all={RUN_ALL_DATASETS}')
print(f'[PARAM] adapter_key={ADAPTER_KEY}')
print(f'[PARAM] review_adapter_key={REVIEW_ADAPTER_KEY}')
print(f'[PARAM] prepared_root={PREPARED_ROOT}')
print(f'[PARAM] dataset_root={dataset_root}')
print(f'[PARAM] output_dir={output_dir}')


In [ ]:
cmd = [sys.executable, 'scripts/audit_ood_oe_quality.py']
if RUN_ALL_DATASETS:
    cmd += ['--all', '--prepared-root', str(PREPARED_ROOT)]
else:
    cmd += ['--dataset-root', str(dataset_root), '--dataset-key', ADAPTER_KEY]
cmd += [
    '--output-dir', str(output_dir),
    '--near-duplicate-hamming', str(NEAR_DUPLICATE_HAMMING),
]
completed = subprocess.run(cmd, check=False, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError(f'Audit failed with exit code {completed.returncode}')

summary_name = 'batch_summary.json' if RUN_ALL_DATASETS else 'summary.json'
summary = json.loads((output_dir / summary_name).read_text(encoding='utf-8'))

if RUN_ALL_DATASETS:
    dataset_rows = list(summary.get('datasets', []))
    dataset_keys = [row.get('dataset_key') for row in dataset_rows]
    if REVIEW_ADAPTER_KEY not in dataset_keys:
        first_with_issues = next((row for row in dataset_rows if int(row.get('issue_count', 0) or 0) > 0), None)
        REVIEW_ADAPTER_KEY = (first_with_issues or dataset_rows[0]).get('dataset_key') if dataset_rows else REVIEW_ADAPTER_KEY
    review_csv = output_dir / REVIEW_ADAPTER_KEY / 'review_decisions.csv'
    review_report = output_dir / REVIEW_ADAPTER_KEY / 'review_report.md'
    print('[AUDIT] batch summary CSV:', output_dir / 'batch_summary.csv')
    print('[AUDIT] selected dataset:', REVIEW_ADAPTER_KEY)
else:
    dataset_rows = [{'dataset_key': ADAPTER_KEY, 'dataset_root': str(dataset_root)}]
    REVIEW_ADAPTER_KEY = ADAPTER_KEY
    review_csv = output_dir / 'review_decisions.csv'
    review_report = output_dir / 'review_report.md'
    print('[AUDIT] selected dataset:', REVIEW_ADAPTER_KEY)

print('[AUDIT] review CSV:', review_csv)
print('[AUDIT] markdown report:', review_report)

try:
    import pandas as pd
    from IPython.display import display

    if RUN_ALL_DATASETS:
        display(pd.read_csv(output_dir / 'batch_summary.csv').head(50))
    if review_csv.exists():
        display(pd.read_csv(review_csv).head(100))
    else:
        print('[AUDIT] No review CSV was written for the selected dataset.')
except Exception as exc:
    print('[AUDIT] Pandas preview unavailable:', exc)


## Human Review

1. Open the printed `review_decisions.csv` for the selected dataset.
2. Inspect the paired paths and `target_path` before taking action.
3. For confirmed cleanup only, set `decision` to `quarantine`.
4. Leave uncertain rows blank, `accept`, or `ignore`.
5. Re-run the apply cell with `APPLY_REVIEW_DECISIONS = True`.

Batch mode writes one CSV per dataset under `<output_dir>/<dataset_key>/review_decisions.csv`. Change `REVIEW_ADAPTER_KEY` in the parameter cell and rerun the audit cell if you want to preview another dataset.


In [ ]:
APPLY_REVIEW_DECISIONS = False

if RUN_ALL_DATASETS:
    dataset_info = next((row for row in dataset_rows if row.get('dataset_key') == REVIEW_ADAPTER_KEY), None)
    active_dataset_root = Path((dataset_info or {}).get('dataset_root') or (PREPARED_ROOT / REVIEW_ADAPTER_KEY))
    decisions_csv = output_dir / REVIEW_ADAPTER_KEY / 'review_decisions.csv'
else:
    active_dataset_root = dataset_root
    decisions_csv = output_dir / 'review_decisions.csv'

quarantine_root = REPO_ROOT / '.runtime_tmp' / 'ood_oe_quality_quarantine' / REVIEW_ADAPTER_KEY

print('[APPLY] dataset:', REVIEW_ADAPTER_KEY)
print('[APPLY] dataset_root:', active_dataset_root)
print('[APPLY] decisions_csv:', decisions_csv)
print('[APPLY] quarantine_root:', quarantine_root)

if not APPLY_REVIEW_DECISIONS:
    print('[APPLY] Closed. Edit review_decisions.csv manually, then set APPLY_REVIEW_DECISIONS=True and rerun this cell.')
elif not decisions_csv.is_file():
    raise FileNotFoundError(f'Decisions CSV not found: {decisions_csv}')
else:
    cmd = [
        sys.executable,
        'scripts/audit_ood_oe_quality.py',
        '--dataset-root', str(active_dataset_root),
        '--apply-decisions', str(decisions_csv),
        '--quarantine-root', str(quarantine_root),
    ]
    completed = subprocess.run(cmd, check=False, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f'Apply failed with exit code {completed.returncode}')


## Complete

Re-run the audit cell after applying decisions to verify the remaining OOD/OE surface. The apply step moves confirmed files to quarantine; it does not delete files and it does not generate quarantine decisions automatically.
